In [ ]:
# -*- coding: utf-8 -*-
"""
spatial_analysis.py

Este script realiza a Análise de Autocorrelação Espacial Local (LISA)
para identificar clusters espaciais de risco de desastres.
Ele generaliza as funcionalidades dos scripts anteriores de análise LISA.

Funcionalidades:s
- Calcula pesos espaciais (Queen, KNN).
- Realiza a análise LISA para um conjunto de variáveis especificadas.
- Adiciona os resultados da análise (Índice de Moran Local, p-valor, significância, clusters)
  ao GeoDataFrame de entrada.
- Salva o GeoDataFrame enriquecido.
"""

# Montar o Google Drive se estiver no Colab e os caminhos forem do Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Descomente se necessário

# Instalar dependências antes de importar
!pip install -r /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/requirements.txt

import pandas as pd
import numpy as np
import geopandas as gpd
from pysal.lib import weights
from pysal.explore import esda
from splot.esda import plot_local_autocorrelation # Para visualização
import logging
import matplotlib.pyplot as plt
import os

# --- Configurações (Idealmente viriam de um arquivo de configuração) ---
# INPUT_GEOPARQUET_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk.geoparquet'
# OUTPUT_GEOPARQUET_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk_lisa_analysis.geoparquet'
# OUTPUT_FIGURES_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/figures/lisa/' # Diretório para salvar as figuras
# DEFAULT_WEIGHT_TYPE = 'queen' # 'queen' ou 'knn'
# DEFAULT_KNN_K = 8 # Número de vizinhos para KNN
# DEFAULT_P_VALUE_THRESHOLD = 0.05
# UFS_TO_FILTER = None # Lista de UFs para filtrar, ex: ['RS'], ou None para todas
# -----------------------------------------------------------------------


# Configuração de Logging
logging.basicConfig(filename='spatial_analysis_log.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Suprimir avisos comuns em análises geoespaciais e pandas
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="pysal")


def calculate_spatial_weights(gdf, weight_type='queen', k=8, id_variable=None):
    """
    Calcula a matriz de pesos espaciais.

    Args:
        gdf (gpd.GeoDataFrame): GeoDataFrame com as geometrias.
        weight_type (str): Tipo de peso ('queen', 'rook', 'knn').
        k (int): Número de vizinhos para KNN.
        id_variable (str, optional): Nome da coluna para usar como ID das observações.
                                     Se None, usa o índice do GeoDataFrame.

    Returns:
        pysal.lib.weights.W: Objeto de pesos espaciais.
    """
    logging.info(f"Calculando pesos espaciais do tipo '{weight_type}'.")
    print(f"Calculando pesos espaciais do tipo '{weight_type}'...") # Notificação ao usuário
    try:
        if weight_type.lower() == 'queen':
            w = weights.Queen.from_dataframe(gdf, idVariable=id_variable)
        elif weight_type.lower() == 'rook':
            w = weights.Rook.from_dataframe(gdf, idVariable=id_variable)
        elif weight_type.lower() == 'knn':
            w = weights.KNN.from_dataframe(gdf, k=k, idVariable=id_variable)
        else:
            logging.error(f"Tipo de peso '{weight_type}' não suportado.")
            print(f"ERRO: Tipo de peso '{weight_type}' não suportado.") # Notificação ao usuário
            raise ValueError(f"Tipo de peso '{weight_type}' não suportado.")

        if w.n_components > 1:
            msg = (f"O grafo de pesos tem {w.n_components} componentes disjuntos (ilhas). "
                   "Isso pode afetar a análise LISA para algumas unidades.")
            logging.warning(msg)
            print(f"AVISO: {msg}") # Notificação ao usuário

        w.transform = 'R'
        logging.info("Pesos espaciais calculados e normalizados por linha.")
        print("Pesos espaciais calculados e normalizados por linha.") # Notificação ao usuário
        return w
    except Exception as e:
        logging.error(f"Erro ao calcular pesos espaciais: {e}")
        print(f"ERRO ao calcular pesos espaciais: {e}") # Notificação ao usuário
        raise


def perform_lisa_analysis(gdf, variables_to_analyze, spatial_weights, p_threshold=0.05):
    """
    Realiza a análise LISA (Local Indicators of Spatial Association) para as variáveis especificadas.

    Args:
        gdf (gpd.GeoDataFrame): GeoDataFrame de entrada.
        variables_to_analyze (list): Lista de nomes de colunas para analisar.
        spatial_weights (pysal.lib.weights.W): Matriz de pesos espaciais pré-calculada.
        p_threshold (float): Limiar de significância para identificar clusters.

    Returns:
        gpd.GeoDataFrame: GeoDataFrame com colunas adicionais dos resultados LISA.
    """
    gdf_analysis = gdf.copy()
    w = spatial_weights

    for var in variables_to_analyze:
        logging.info(f"Iniciando análise LISA para a variável: {var}")
        print(f"\nIniciando análise LISA para a variável: {var}...") # Notificação ao usuário
        if var not in gdf_analysis.columns:
            msg = f"Variável '{var}' não encontrada no GeoDataFrame. Pulando."
            logging.warning(msg)
            print(f"AVISO: {msg}") # Notificação ao usuário
            continue

        if gdf_analysis[var].isnull().any():
            msg = (f"Variável '{var}' contém {gdf_analysis[var].isnull().sum()} valores NaN. "
                   "Preenchendo com 0 para análise LISA. Considere uma imputação mais sofisticada se necessário.")
            logging.warning(msg)
            print(f"AVISO: {msg}") # Notificação ao usuário
            gdf_analysis[var] = gdf_analysis[var].fillna(0)

        if gdf_analysis[var].nunique() <= 1:
            msg = f"Variável '{var}' tem variância insuficiente (nunique <= 1). Pulando análise LISA."
            logging.warning(msg)
            print(f"AVISO: {msg}") # Notificação ao usuário
            gdf_analysis[f'{var}_lisa_I'] = np.nan
            gdf_analysis[f'{var}_lisa_p_sim'] = np.nan
            gdf_analysis[f'{var}_lisa_sig'] = False
            gdf_analysis[f'{var}_lisa_q'] = 'N/A'
            gdf_analysis[f'{var}_lisa_labels'] = '0 ns'
            gdf_analysis[f'w_{var}'] = np.nan
            continue

        try:
            moran_global = esda.Moran(gdf_analysis[var], w, permutations=999)
            logging.info(f"Moran Global para '{var}': I={moran_global.I:.4f}, p-valor_sim={moran_global.p_sim:.4f}")
            print(f"  Moran Global para '{var}': I={moran_global.I:.4f}, p-valor_sim={moran_global.p_sim:.4f}") # Notificação ao usuário

            lisa = esda.Moran_Local(gdf_analysis[var], w, permutations=999, seed=12345)

            gdf_analysis[f'{var}_lisa_I'] = lisa.Is
            gdf_analysis[f'{var}_lisa_p_sim'] = lisa.p_sim
            gdf_analysis[f'{var}_lisa_sig'] = lisa.p_sim < p_threshold

            q_labels_map = {1: 'HH (Alto-Alto)', 2: 'LH (Baixo-Alto)', 3: 'LL (Baixo-Baixo)', 4: 'HL (Alto-Baixo)'}
            gdf_analysis[f'{var}_lisa_q_code'] = lisa.q
            gdf_analysis[f'{var}_lisa_q'] = gdf_analysis[f'{var}_lisa_q_code'].map(q_labels_map).fillna('N/A')

            spot_labels = ['0 ns (Não Significativo)',
                           '1 HH (Hot Spot)',
                           '2 LH (Anel Baixo-Alto)',
                           '3 LL (Cold Spot)',
                           '4 HL (Anel Alto-Baixo)']

            cluster_labels = np.zeros(len(gdf_analysis), dtype=int)
            significant_filter = gdf_analysis[f'{var}_lisa_sig'] == True

            cluster_labels[significant_filter & (lisa.q == 1)] = 1
            cluster_labels[significant_filter & (lisa.q == 2)] = 2
            cluster_labels[significant_filter & (lisa.q == 3)] = 3
            cluster_labels[significant_filter & (lisa.q == 4)] = 4

            gdf_analysis[f'{var}_lisa_labels_code'] = cluster_labels
            gdf_analysis[f'{var}_lisa_labels'] = gdf_analysis[f'{var}_lisa_labels_code'].apply(lambda x: spot_labels[x])

            gdf_analysis[f'w_{var}'] = weights.lag_spatial(w, gdf_analysis[var])

            logging.info(f"Análise LISA concluída para '{var}'.")
            print(f"  Análise LISA concluída para '{var}'.") # Notificação ao usuário

        except Exception as e:
            logging.error(f"Erro durante a análise LISA para '{var}': {e}")
            print(f"  ERRO durante a análise LISA para '{var}': {e}") # Notificação ao usuário
            gdf_analysis[f'{var}_lisa_I'] = np.nan
            gdf_analysis[f'{var}_lisa_p_sim'] = np.nan
            gdf_analysis[f'{var}_lisa_sig'] = False
            gdf_analysis[f'{var}_lisa_q'] = 'Erro'
            gdf_analysis[f'{var}_lisa_labels'] = 'Erro'
            gdf_analysis[f'w_{var}'] = np.nan
            continue

    return gdf_analysis


def plot_lisa_results(gdf, variable_name, lisa_labels_col, output_path, filename_suffix=""):
    """
    Plota e salva o mapa de clusters LISA para uma variável.
    """
    if lisa_labels_col not in gdf.columns:
        logging.warning(f"Coluna de rótulos LISA '{lisa_labels_col}' não encontrada para '{variable_name}'. Pular plotagem.")
        return

    logging.info(f"Plotando mapa de clusters LISA para '{variable_name}'.")
    print(f"Plotando mapa de clusters LISA para '{variable_name}'...") # Notificação ao usuário

    cluster_colors = {
        '0 ns (Não Significativo)': '#eeeeee',
        '1 HH (Hot Spot)': '#FF0000',
        '2 LH (Anel Baixo-Alto)': '#FFA500',
        '3 LL (Cold Spot)': '#0000FF',
        '4 HL (Anel Alto-Baixo)': '#ADD8E6',
        'Erro': '#000000'
    }

    try:
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))
        colors_mapped = gdf[lisa_labels_col].map(cluster_colors).fillna('#000000')

        gdf.plot(
            color=colors_mapped,
            ax=ax,
            edgecolor='black',
            linewidth=0.5,
            legend=False
        )

        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=color, edgecolor='k', label=label)
                           for label, color in cluster_colors.items() if label in gdf[lisa_labels_col].unique()]

        if legend_elements:
            ax.legend(handles=legend_elements, title="Clusters LISA", loc='lower right', fontsize='small')

        ax.set_title(f"Clusters LISA para: {variable_name}", fontsize=15)
        ax.set_axis_off()

        os.makedirs(output_path, exist_ok=True)
        filepath = os.path.join(output_path, f"lisa_map_{variable_name}{filename_suffix}.png")
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close(fig)
        logging.info(f"Mapa LISA para '{variable_name}' salvo em: {filepath}")
        print(f"  Mapa LISA para '{variable_name}' salvo em: {filepath}") # Notificação ao usuário

    except Exception as e:
        logging.error(f"Erro ao plotar mapa LISA para '{variable_name}': {e}")
        print(f"  ERRO ao plotar mapa LISA para '{variable_name}': {e}") # Notificação ao usuário


def main_spatial_analysis(config):
    """
    Função principal para executar o pipeline de análise espacial.
    """
    logging.info("--- Iniciando Pipeline de Análise Espacial (LISA) ---")
    print("--- Iniciando Pipeline de Análise Espacial (LISA) ---") # Notificação ao usuário

    input_path = config['INPUT_GEOPARQUET_PATH']
    output_path = config['OUTPUT_GEOPARQUET_PATH']

    logging.info(f"Carregando dados de: {input_path}")
    print(f"Carregando dados de: {input_path}...") # Notificação ao usuário
    try:
        gdf = gpd.read_parquet(input_path)
        if gdf.empty:
            msg = "GeoDataFrame carregado está vazio. Abortando."
            logging.error(msg)
            print(f"ERRO: {msg}") # Notificação ao usuário
            return
        if 'geometry' not in gdf.columns or gdf.geometry.isnull().all():
             msg = "Coluna de geometria ausente ou vazia. Abortando."
             logging.error(msg)
             print(f"ERRO: {msg}") # Notificação ao usuário
             return
        gdf = gdf.set_geometry('geometry')
        logging.info(f"Dados carregados: {gdf.shape[0]} linhas, {gdf.shape[1]} colunas.")
        print(f"Dados carregados: {gdf.shape[0]} linhas, {gdf.shape[1]} colunas.") # Notificação ao usuário
    except Exception as e:
        logging.error(f"Erro ao carregar o arquivo GeoParquet de entrada: {e}")
        print(f"ERRO ao carregar o arquivo GeoParquet de entrada: {e}") # Notificação ao usuário
        return

    if config.get('UFS_TO_FILTER'):
        ufs = config['UFS_TO_FILTER']
        logging.info(f"Filtrando GeoDataFrame para UFs: {ufs}")
        print(f"Filtrando GeoDataFrame para UFs: {ufs}...") # Notificação ao usuário
        gdf = gdf[gdf['SIGLA_UF'].isin(ufs)]
        if gdf.empty:
            msg = f"GeoDataFrame vazio após filtrar por UFs: {ufs}. Abortando."
            logging.error(msg)
            print(f"ERRO: {msg}") # Notificação ao usuário
            return
        logging.info(f"GeoDataFrame filtrado: {gdf.shape[0]} linhas.")
        print(f"GeoDataFrame filtrado: {gdf.shape[0]} linhas.") # Notificação ao usuário

    gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy() # Adicionado .copy() para evitar SettingWithCopyWarning
    if gdf.empty:
        msg = "GeoDataFrame vazio após remover geometrias inválidas/vazias. Abortando."
        logging.error(msg)
        print(f"ERRO: {msg}") # Notificação ao usuário
        return

    try:
        id_col_for_weights = 'CD_MUN' if 'CD_MUN' in gdf.columns else None
        if id_col_for_weights and gdf[id_col_for_weights].duplicated().any():
            logging.warning(f"Coluna ID '{id_col_for_weights}' tem valores duplicados. Usando índice do GeoDataFrame para pesos.")
            print(f"AVISO: Coluna ID '{id_col_for_weights}' tem valores duplicados. Usando índice do GeoDataFrame para pesos.") # Notificação ao usuário
            id_col_for_weights = None

        spatial_weights = calculate_spatial_weights(gdf,
                                                weight_type=config.get('DEFAULT_WEIGHT_TYPE', 'queen'),
                                                k=config.get('DEFAULT_KNN_K', 8),
                                                id_variable=id_col_for_weights)
    except Exception as e:
        logging.error(f"Falha ao calcular pesos espaciais. Abortando análise. Erro: {e}")
        print(f"ERRO: Falha ao calcular pesos espaciais. Abortando análise. Erro: {e}") # Notificação ao usuário
        return

    variables_to_analyze = config.get('VARIABLES_FOR_LISA', [])
    if not variables_to_analyze:
        logging.warning("Nenhuma variável especificada para análise LISA em 'VARIABLES_FOR_LISA'. Tentando identificar automaticamente colunas de contagem e taxa.")
        print("AVISO: Nenhuma variável especificada para análise LISA. Tentando identificar automaticamente...") # Notificação ao usuário
        for col in gdf.columns:
            if ("COUNT" in col.upper() or "POR_10K_HAB" in col.upper()) and pd.api.types.is_numeric_dtype(gdf[col]):
                if not ("_LISA_" in col.upper() or "_P_SIM" in col.upper() or "_SIG" in col.upper() or "_Q" in col.upper() or "W_" in col.upper() or "_LABELS" in col.upper()):
                    variables_to_analyze.append(col)
        if not variables_to_analyze:
             msg = "Não foi possível identificar variáveis numéricas para análise LISA. Verifique as colunas do GeoDataFrame."
             logging.error(msg)
             print(f"ERRO: {msg}") # Notificação ao usuário
             return
        logging.info(f"Variáveis identificadas para análise: {variables_to_analyze}")
        print(f"Variáveis identificadas para análise: {variables_to_analyze}") # Notificação ao usuário

    gdf_with_lisa = perform_lisa_analysis(gdf, variables_to_analyze, spatial_weights,
                                          p_threshold=config.get('DEFAULT_P_VALUE_THRESHOLD', 0.05))

    output_figures_path = config.get('OUTPUT_FIGURES_PATH')
    if output_figures_path:
        os.makedirs(output_figures_path, exist_ok=True)
        print(f"\nSalvando figuras LISA em: {output_figures_path}") # Notificação ao usuário
        for var in variables_to_analyze:
            lisa_labels_col_name = f'{var}_lisa_labels'
            if lisa_labels_col_name in gdf_with_lisa.columns:
                plot_lisa_results(gdf_with_lisa, var, lisa_labels_col_name, output_figures_path)
            else:
                logging.warning(f"Coluna de rótulos LISA '{lisa_labels_col_name}' não encontrada para '{var}' após a análise. Não será plotada.")
                print(f"AVISO: Coluna de rótulos LISA '{lisa_labels_col_name}' não encontrada para '{var}'. Mapa não gerado.") # Notificação ao usuário


    logging.info(f"Salvando GeoDataFrame com resultados LISA em: {output_path}")
    print(f"\nSalvando GeoDataFrame com resultados LISA em: {output_path}...") # Notificação ao usuário
    try:
        gdf_with_lisa.to_parquet(output_path)
        logging.info("GeoDataFrame salvo com sucesso.")
        print(f"SUCESSO: GeoDataFrame com resultados LISA salvo em: {output_path}") # Notificação ao usuário
    except Exception as e:
        logging.error(f"Erro ao salvar o GeoDataFrame com resultados LISA: {e}")
        print(f"ERRO ao salvar o GeoDataFrame com resultados LISA: {e}") # Notificação ao usuário

    logging.info("--- Pipeline de Análise Espacial (LISA) Concluído ---")
    print("--- Pipeline de Análise Espacial (LISA) Concluído ---") # Notificação ao usuário


if __name__ == '__main__':
    BASE_PROJECT_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr'

    config_spatial_analysis = {
        'INPUT_GEOPARQUET_PATH': os.path.join(BASE_PROJECT_PATH, '02_silver/climate_risk.geoparquet'),
        'OUTPUT_GEOPARQUET_PATH': os.path.join(BASE_PROJECT_PATH, '02_silver/climate_risk_lisa_analysis.geoparquet'),
        'OUTPUT_FIGURES_PATH': os.path.join(BASE_PROJECT_PATH, '03_gold/figures/lisa/'),
        'DEFAULT_WEIGHT_TYPE': 'queen',
        'DEFAULT_KNN_K': 8,
        'DEFAULT_P_VALUE_THRESHOLD': 0.05,
        'UFS_TO_FILTER': None,
        'VARIABLES_FOR_LISA': [
         'HISTORIC_COUNT',
            'LAST10_YEARS_COUNT',
            'LAST05_YEARS_COUNT',
            'LAST02_YEARS_COUNT',
            'HISTORIC_COUNT_POR_10K_HAB', # Exemplo de taxa
            'LAST10_YEARS_COUNT_POR_10K_HAB',
            'LAST05_YEARS_COUNT_POR_10K_HAB',
            'LAST02_YEARS_COUNT_POR_10K_HAB',
            'LAST10_YEARS_COUNT_11321', # 11321 - Deslizamentos
            'LAST05_YEARS_COUNT_11321',
            'LAST02_YEARS_COUNT_11321',
            'LAST10_YEARS_COUNT_11331', # 11331 - Corridas de Massa - Solo/Lama
            'LAST05_YEARS_COUNT_11331',
            'LAST02_YEARS_COUNT_11331',
            'LAST10_YEARS_COUNT_11332', # 11332 - Corridas de Massa - Rocha/detrito
            'LAST05_YEARS_COUNT_11332',
            'LAST02_YEARS_COUNT_11332',
            'LAST10_YEARS_COUNT_12100', # 12100 - Inundações
            'LAST05_YEARS_COUNT_12100',
            'LAST02_YEARS_COUNT_12100',
            'LAST10_YEARS_COUNT_12200', # 12200 - Enxurradas
            'LAST05_YEARS_COUNT_12200',
            'LAST02_YEARS_COUNT_12200',
            'LAST10_YEARS_COUNT_12300', # 12300 - Alagamentos
            'LAST05_YEARS_COUNT_12300',
            'LAST02_YEARS_COUNT_12300',
            'LAST10_YEARS_COUNT_13213',  # 13213 - Tempestade Local/Convectiva - Granizo
            'LAST05_YEARS_COUNT_13213',
            'LAST02_YEARS_COUNT_13213',
            'LAST10_YEARS_COUNT_13214',  # 13214 - Tempestade Local/Convectiva - Chuvas Intensas
            'LAST05_YEARS_COUNT_13214',
            'LAST02_YEARS_COUNT_13214',
            'LAST10_YEARS_COUNT_13215',  # 13215 - Tempestade Local/Convectiva - Vendaval
            'LAST05_YEARS_COUNT_13215',
            'LAST02_YEARS_COUNT_13215',
            'LAST10_YEARS_COUNT_24200',  # 24200 - Rompimento/colapso de barragens
            'LAST05_YEARS_COUNT_24200',
            'LAST02_YEARS_COUNT_24200',
            'LAST10_YEARS_COUNT_POR_10K_HAB_11321', # 11321 - Deslizamentos
            'LAST05_YEARS_COUNT_POR_10K_HAB_11321',
            'LAST02_YEARS_COUNT_POR_10K_HAB_11321',
            'LAST10_YEARS_COUNT_POR_10K_HAB_11331', # 11331 - Corridas de Massa - Solo/Lama
            'LAST05_YEARS_COUNT_POR_10K_HAB_11331',
            'LAST02_YEARS_COUNT_POR_10K_HAB_11331',
            'LAST10_YEARS_COUNT_POR_10K_HAB_11332', # 11332 - Corridas de Massa - Rocha/detrito
            'LAST05_YEARS_COUNT_POR_10K_HAB_11332',
            'LAST02_YEARS_COUNT_POR_10K_HAB_11332',
            'LAST10_YEARS_COUNT_POR_10K_HAB_12100', # 12100 - Inundações
            'LAST05_YEARS_COUNT_POR_10K_HAB_12100',
            'LAST02_YEARS_COUNT_POR_10K_HAB_12100',
            'LAST10_YEARS_COUNT_POR_10K_HAB_12200', # 12200 - Enxurradas
            'LAST05_YEARS_COUNT_POR_10K_HAB_12200',
            'LAST02_YEARS_COUNT_POR_10K_HAB_12200',
            'LAST10_YEARS_COUNT_POR_10K_HAB_12300', # 12300 - Alagamentos
            'LAST05_YEARS_COUNT_POR_10K_HAB_12300',
            'LAST02_YEARS_COUNT_POR_10K_HAB_12300',
            'LAST10_YEARS_COUNT_POR_10K_HAB_13213',  # 13213 - Tempestade Local/Convectiva - Granizo
            'LAST05_YEARS_COUNT_POR_10K_HAB_13213',
            'LAST02_YEARS_COUNT_POR_10K_HAB_13213',
            'LAST10_YEARS_COUNT_POR_10K_HAB_13214',  # 13214 - Tempestade Local/Convectiva - Chuvas Intensas
            'LAST05_YEARS_COUNT_POR_10K_HAB_13214',
            'LAST02_YEARS_COUNT_POR_10K_HAB_13214',
            'LAST10_YEARS_COUNT_POR_10K_HAB_13215',  # 13215 - Tempestade Local/Convectiva - Vendaval
            'LAST05_YEARS_COUNT_POR_10K_HAB_13215',
            'LAST02_YEARS_COUNT_POR_10K_HAB_13215',
            'LAST10_YEARS_COUNT_POR_10K_HAB_24200',  # 24200 - Rompimento/colapso de barragens
            'LAST05_YEARS_COUNT_POR_10K_HAB_24200',
            'LAST02_YEARS_COUNT_POR_10K_HAB_24200',
        ]
    }

    os.makedirs(os.path.dirname(config_spatial_analysis['OUTPUT_GEOPARQUET_PATH']), exist_ok=True)
    if config_spatial_analysis.get('OUTPUT_FIGURES_PATH'):
        os.makedirs(config_spatial_analysis['OUTPUT_FIGURES_PATH'], exist_ok=True)

    main_spatial_analysis(config_spatial_analysis)
    # print("Exemplo de execução do main_spatial_analysis (comentado).")
    # print("Descomente a linha `main_spatial_analysis(config_spatial_analysis)` e ajuste os caminhos e variáveis para executar.")

Mounted at /content/drive
INFO: pip is looking at multiple versions of h3pandas to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.6 MB/s eta 0:00:00
  Created wheel for h3pandas: filename=h3pandas-0.2.6-py3-none-any.whl size=19688 sha256=381119245ac3315367e04fccb3511826ae60d896c735817f0d01da5e9da81128
  Stored in directory: /root/.cache/pip/wheels/b8/92/37/51119668d260605aa3e77ff7bc342ad2add69eaac0d77d5744
Successfully built h3pandas


/usr/local/lib/python3.12/dist-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


--- Iniciando Pipeline de Análise Espacial (LISA) ---
Carregando dados de: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk.geoparquet...
Dados carregados: 499 linhas, 521 colunas.
Calculando pesos espaciais do tipo 'queen'...
Pesos espaciais calculados e normalizados por linha.

Iniciando análise LISA para a variável: HISTORIC_COUNT...
  Moran Global para 'HISTORIC_COUNT': I=0.2717, p-valor_sim=0.0010
  Análise LISA concluída para 'HISTORIC_COUNT'.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT...
  Moran Global para 'LAST10_YEARS_COUNT': I=0.1080, p-valor_sim=0.0010
  Análise LISA concluída para 'LAST10_YEARS_COUNT'.

Iniciando análise LISA para a variável: LAST05_YEARS_COUNT...
  Moran Global para 'LAST05_YEARS_COUNT': I=0.1229, p-valor_sim=0.0010
  Análise LISA concluída para 'LAST05_YEARS_COUNT'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT...
  Moran Global para 'LAST02_YEARS_COUNT': I=0.1899, p-valor_sim=0.

  Moran Global para 'LAST05_YEARS_COUNT_11332': I=-0.0035, p-valor_sim=0.0320
  Análise LISA concluída para 'LAST05_YEARS_COUNT_11332'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_11332...
AVISO: Variável 'LAST02_YEARS_COUNT_11332' tem variância insuficiente (nunique <= 1). Pulando análise LISA.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT_12100...
  Moran Global para 'LAST10_YEARS_COUNT_12100': I=0.2529, p-valor_sim=0.0010
  Análise LISA concluída para 'LAST10_YEARS_COUNT_12100'.

Iniciando análise LISA para a variável: LAST05_YEARS_COUNT_12100...
  Moran Global para 'LAST05_YEARS_COUNT_12100': I=0.1889, p-valor_sim=0.0010
  Análise LISA concluída para 'LAST05_YEARS_COUNT_12100'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_12100...
  Moran Global para 'LAST02_YEARS_COUNT_12100': I=0.1010, p-valor_sim=0.0030
  Análise LISA concluída para 'LAST02_YEARS_COUNT_12100'.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT_12200...
  Mora

  Análise LISA concluída para 'LAST05_YEARS_COUNT_24200'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_24200...
AVISO: Variável 'LAST02_YEARS_COUNT_24200' tem variância insuficiente (nunique <= 1). Pulando análise LISA.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT_POR_10K_HAB_11321...
  Moran Global para 'LAST10_YEARS_COUNT_POR_10K_HAB_11321': I=0.0324, p-valor_sim=0.0410
  Análise LISA concluída para 'LAST10_YEARS_COUNT_POR_10K_HAB_11321'.

Iniciando análise LISA para a variável: LAST05_YEARS_COUNT_POR_10K_HAB_11321...
  Moran Global para 'LAST05_YEARS_COUNT_POR_10K_HAB_11321': I=0.0354, p-valor_sim=0.0350
  Análise LISA concluída para 'LAST05_YEARS_COUNT_POR_10K_HAB_11321'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_POR_10K_HAB_11321...
  Moran Global para 'LAST02_YEARS_COUNT_POR_10K_HAB_11321': I=0.0618, p-valor_sim=0.0230
  Análise LISA concluída para 'LAST02_YEARS_COUNT_POR_10K_HAB_11321'.

Iniciando análise LISA para a variável: LAST

  Análise LISA concluída para 'LAST02_YEARS_COUNT_POR_10K_HAB_11331'.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT_POR_10K_HAB_11332...
  Moran Global para 'LAST10_YEARS_COUNT_POR_10K_HAB_11332': I=-0.0024, p-valor_sim=0.3800
  Análise LISA concluída para 'LAST10_YEARS_COUNT_POR_10K_HAB_11332'.

Iniciando análise LISA para a variável: LAST05_YEARS_COUNT_POR_10K_HAB_11332...
  Moran Global para 'LAST05_YEARS_COUNT_POR_10K_HAB_11332': I=-0.0035, p-valor_sim=0.0470
  Análise LISA concluída para 'LAST05_YEARS_COUNT_POR_10K_HAB_11332'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_POR_10K_HAB_11332...
AVISO: Variável 'LAST02_YEARS_COUNT_POR_10K_HAB_11332' tem variância insuficiente (nunique <= 1). Pulando análise LISA.

Iniciando análise LISA para a variável: LAST10_YEARS_COUNT_POR_10K_HAB_12100...
  Moran Global para 'LAST10_YEARS_COUNT_POR_10K_HAB_12100': I=0.0206, p-valor_sim=0.1260
  Análise LISA concluída para 'LAST10_YEARS_COUNT_POR_10K_HAB_12100'.

Inicia

  Moran Global para 'LAST10_YEARS_COUNT_POR_10K_HAB_24200': I=0.0108, p-valor_sim=0.0220
  Análise LISA concluída para 'LAST10_YEARS_COUNT_POR_10K_HAB_24200'.

Iniciando análise LISA para a variável: LAST05_YEARS_COUNT_POR_10K_HAB_24200...
  Moran Global para 'LAST05_YEARS_COUNT_POR_10K_HAB_24200': I=-0.0029, p-valor_sim=0.1160
  Análise LISA concluída para 'LAST05_YEARS_COUNT_POR_10K_HAB_24200'.

Iniciando análise LISA para a variável: LAST02_YEARS_COUNT_POR_10K_HAB_24200...
AVISO: Variável 'LAST02_YEARS_COUNT_POR_10K_HAB_24200' tem variância insuficiente (nunique <= 1). Pulando análise LISA.

Salvando figuras LISA em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/figures/lisa/
Plotando mapa de clusters LISA para 'HISTORIC_COUNT'...
  Mapa LISA para 'HISTORIC_COUNT' salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/figures/lisa/lisa_map_HISTORIC_COUNT.png
Plotando mapa de clusters LISA para 'LAST10_YEARS_COUNT'...

In [ ]:
climate_risk_lisa_analysis = gpd.read_parquet('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk_lisa_analysis.geoparquet')
print(climate_risk_lisa_analysis.shape);climate_risk_lisa_analysis.head(3)

(499, 1057)


,CD_MUN,geometry,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,SIGLA_UF,CD_REGIA,...,LAST05_YEARS_COUNT_POR_10K_HAB_24200_lisa_q,LAST05_YEARS_COUNT_POR_10K_HAB_24200_lisa_labels_code,LAST05_YEARS_COUNT_POR_10K_HAB_24200_lisa_labels,w_LAST05_YEARS_COUNT_POR_10K_HAB_24200,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_I,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_p_sim,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_sig,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_q,LAST02_YEARS_COUNT_POR_10K_HAB_24200_lisa_labels,w_LAST02_YEARS_COUNT_POR_10K_HAB_24200
0,4300001,"POLYGON ((-52.62752 -32.15022, -52.62816 -32.1...",N/A,N/A,N/A,None,43,Rio Grande do Sul,RS,4,...,LL (Baixo-Baixo),3,3 LL (Cold Spot),0.0,NaN,NaN,False,N/A,0 ns,NaN
1,4300002,"POLYGON ((-51.31995 -30.14806, -51.31993 -30.1...",N/A,N/A,N/A,None,43,Rio Grande do Sul,RS,4,...,LL (Baixo-Baixo),3,3 LL (Cold Spot),0.0,NaN,NaN,False,N/A,0 ns,NaN
2,4300034,"POLYGON ((-54.21884 -31.82901, -54.23601 -31.8...",430010,Bagé,4302,Pelotas,43,Rio Grande do Sul,RS,4,...,LL (Baixo-Baixo),3,3 LL (Cold Spot),0.0,NaN,NaN,False,N/A,0 ns,NaN


In [ ]:
climate_risk_lisa_analysis.drop(['geometry'], axis=1).to_csv('/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/02_silver/climate_risk_lisa_analysis.csv', sep=';', index=False)

In [ ]:
climate_risk_lisa_analysis[['NM_MUN_SEM_ACENTO',
                            'CENSO_2020_POP',
                            'HISTORIC_COUNT',
                            'HISTORIC_COUNT_lisa_I',
                            'LAST10_YEARS_COUNT',
                            'LAST10_YEARS_COUNT_lisa_I',
                            'LAST05_YEARS_COUNT',
                            'LAST05_YEARS_COUNT_lisa_I',
                            'LAST02_YEARS_COUNT',
                            'LAST02_YEARS_COUNT_lisa_I',
                            ]]

,NM_MUN_SEM_ACENTO,CENSO_2020_POP,HISTORIC_COUNT,HISTORIC_COUNT_lisa_I,LAST10_YEARS_COUNT,LAST10_YEARS_COUNT_lisa_I,LAST05_YEARS_COUNT,LAST05_YEARS_COUNT_lisa_I,LAST02_YEARS_COUNT,LAST02_YEARS_COUNT_lisa_I
0,"AREA OPERACIONAL ""LAGOA MIRIM""",0,0.0,-2.701290,0.0,-3.645235,0.0,-4.403226,0.0,-4.767178
1,"AREA OPERACIONAL ""LAGOA DOS PATOS""",0,0.0,-0.270280,0.0,-1.516703,0.0,-1.472624,0.0,-1.771369
2,ACEGUA,4170,18.0,-0.327520,9.0,-0.431688,4.0,-0.635487,3.0,-0.065656
3,AGUA SANTA,3912,23.0,0.000341,9.0,-0.005370,5.0,0.172478,2.0,-0.054207
4,AGUDO,16041,37.0,0.658998,21.0,0.107850,17.0,0.853071,5.0,0.593546
...,...,...,...,...,...,...,...,...,...,...
494,VISTA ALEGRE DO PRATA,1590,12.0,0.775963,6.0,0.487582,4.0,0.540822,2.0,0.536906
495,VISTA GAUCHA,2783,24.0,0.112025,11.0,-0.056954,8.0,0.006520,3.0,-0.021478
496,VITORIA DAS MISSOES,3260,16.0,-0.382005,7.0,-0.160535,5.0,0.018568,2.0,-0.101496
497,WESTFALIA,3098,10.0,0.619023,6.0,-0.169214,5.0,-0.443163,1.0,-0.633788


In [ ]:
climate_risk_lisa_analysis[['NM_MUN_SEM_ACENTO',
                            'CENSO_2020_POP',
                            'HISTORIC_COUNT_POR_10K_HAB',
                            'HISTORIC_COUNT_POR_10K_HAB_lisa_I',
                            'LAST10_YEARS_COUNT_POR_10K_HAB',
                            'LAST10_YEARS_COUNT_POR_10K_HAB_lisa_I',
                            'LAST05_YEARS_COUNT_POR_10K_HAB',
                            'LAST05_YEARS_COUNT_POR_10K_HAB_lisa_I',
                            'LAST02_YEARS_COUNT_POR_10K_HAB',
                            'LAST02_YEARS_COUNT_POR_10K_HAB_lisa_I',
                            ]]

,NM_MUN_SEM_ACENTO,CENSO_2020_POP,HISTORIC_COUNT_POR_10K_HAB,HISTORIC_COUNT_POR_10K_HAB_lisa_I,LAST10_YEARS_COUNT_POR_10K_HAB,LAST10_YEARS_COUNT_POR_10K_HAB_lisa_I,LAST05_YEARS_COUNT_POR_10K_HAB,LAST05_YEARS_COUNT_POR_10K_HAB_lisa_I,LAST02_YEARS_COUNT_POR_10K_HAB,LAST02_YEARS_COUNT_POR_10K_HAB_lisa_I
0,"AREA OPERACIONAL ""LAGOA MIRIM""",0,0.000000,1.217959,0.000000,1.027856,0.000000,1.008965,0.000000,0.674099
1,"AREA OPERACIONAL ""LAGOA DOS PATOS""",0,0.000000,1.100815,0.000000,0.931329,0.000000,0.910950,0.000000,0.524694
2,ACEGUA,4170,43.165468,-0.006732,21.582734,-0.004511,9.592326,-0.079554,7.194245,0.056446
3,AGUA SANTA,3912,58.793456,0.400612,23.006135,0.081085,12.781186,-0.162485,5.112474,-0.200875
4,AGUDO,16041,23.065894,-0.267365,13.091453,-0.152637,10.597843,-0.179411,3.117013,-0.337989
...,...,...,...,...,...,...,...,...,...,...
494,VISTA ALEGRE DO PRATA,1590,75.471698,-0.669429,37.735849,-0.673102,25.157233,-0.569890,12.578616,-0.866051
495,VISTA GAUCHA,2783,86.237873,0.270243,39.525692,0.321091,28.745958,0.191629,10.779734,0.185518
496,VITORIA DAS MISSOES,3260,49.079755,-0.064607,21.472393,0.008680,15.337423,-0.019686,6.134969,0.006696
497,WESTFALIA,3098,32.278890,0.075851,19.367334,-0.034739,16.139445,0.037655,3.227889,-0.030372


In [ ]:
for i in climate_risk_lisa_analysis.columns:
  print("'"+i+"'"+",")

'CD_MUN',
'geometry',
'CD_RGI',
'NM_RGI',
'CD_RGINT',
'NM_RGINT',
'CD_UF',
'NM_UF',
'SIGLA_UF',
'CD_REGIA',
'NM_REGIA',
'SIGLA_RG',
'CD_CONCU',
'NM_CONCU',
'AREA_KM2',
'CENSO_2020_POP',
'NM_MUN_SEM_ACENTO',
'HISTORIC_COUNT',
'LAST10_YEARS_COUNT',
'LAST05_YEARS_COUNT',
'LAST02_YEARS_COUNT',
'HISTORIC_COUNT_POR_10K_HAB',
'LAST10_YEARS_COUNT_POR_10K_HAB',
'LAST05_YEARS_COUNT_POR_10K_HAB',
'LAST02_YEARS_COUNT_POR_10K_HAB',
'HISTORIC_COUNT_11110',
'HISTORIC_COUNT_11120',
'HISTORIC_COUNT_11311',
'HISTORIC_COUNT_11312',
'HISTORIC_COUNT_11313',
'HISTORIC_COUNT_11314',
'HISTORIC_COUNT_11321',
'HISTORIC_COUNT_11331',
'HISTORIC_COUNT_11332',
'HISTORIC_COUNT_11340',
'HISTORIC_COUNT_11410',
'HISTORIC_COUNT_11420',
'HISTORIC_COUNT_11431',
'HISTORIC_COUNT_11432',
'HISTORIC_COUNT_11433',
'HISTORIC_COUNT_12100',
'HISTORIC_COUNT_12200',
'HISTORIC_COUNT_12300',
'HISTORIC_COUNT_13111',
'HISTORIC_COUNT_13112',
'HISTORIC_COUNT_13120',
'HISTORIC_COUNT_13211',
'HISTORIC_COUNT_13212',
'HISTORIC_COUNT_13213',
'